#### Preparing Data

In [ ]:
# import the necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import keras

In [ ]:
# reading csv data
columns = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "total_LCP",
    "emissions",
    "PFAS_eff",
    "PFAS_ads",
    ]

df1 = pd.read_csv("./training_data/GAC_60k_CORRECTED_26-03-2026.csv", usecols=columns)
df = df1.dropna(ignore_index=True)
# df.tail()

In [ ]:
# count rows before cleaning
print(f"Number of rows before cleaning: {len(df)}")

In [ ]:
epsilon = 1e-20
# cleaning data
df.loc[:, 'PFAS_eff'] = df['PFAS_eff'].clip(lower=0)
df = df[(df['total_LCP'] >= 0)].copy()
df['total_LCP'] = np.log(df['total_LCP'])
df['emissions'] = np.log(df['emissions'])
df['PFAS_eff'] = np.log(df['PFAS_eff'] + epsilon)

# bounds
print(df.min())
print(df.max())

In [ ]:
# count rows after cleaning
print(f"Number of rows after cleaning: {len(df)}")

In [ ]:
# separate the data into inputs and outputs
inputs = ["capacity", "diameter", "column_height", "porosity", "ads_time", "C_o"]
outputs = [
    "total_LCP",
    "emissions",
    "PFAS_eff",
]
dfin = df[inputs]
dfout = df[outputs]

In [ ]:
# define a function to scale the data using min-max scaling
def min_max_scaler(data, data_min, data_max):
    return (data - data_min) /(data_max - data_min)

# define a function to reverse the scaling
def reverse_min_max_scaler(scaled_data, data_min, data_max):
    return scaled_data * (data_max - data_min) + data_min

In [ ]:
# scale the data
x_data_min = dfin.min()
x_data_max = dfin.max()
y_data_min = dfout.min()
y_data_max = dfout.max()

x_data_scaled = min_max_scaler(dfin, x_data_min, x_data_max)
y_data_scaled = min_max_scaler(dfout, y_data_min, y_data_max)

x_scaled_min = x_data_scaled.min()
x_scaled_max = x_data_scaled.max()
y_scaled_min = y_data_scaled.min()
y_scaled_max = y_data_scaled.max()


#### Optimise Model with pyomo

In [ ]:
import pyomo.environ as pyo
# omlt for interfacing our neural network with pyomo
from Custom_scaling import MinMaxScaling
from omlt import OffsetScaling, OmltBlock
from omlt.io.keras import keras_reader, load_keras_sequential
from omlt.neuralnet import (
    FullSpaceNNFormulation,
    FullSpaceSmoothNNFormulation,
    ReducedSpaceSmoothNNFormulation,
    ReluBigMFormulation,
    ReluComplementarityFormulation,
    ReluPartitionFormulation,
)
from omlt.neuralnet.activations import ComplementarityReLUActivation

In [ ]:
# # Check solver availability
# for solver in ["glpk", "cbc", "ipopt", "baron", "cplex", "appsi_maingo", "gurobi"]:
#     print(solver, pyo.SolverFactory(solver).available())

In [ ]:
# unscaling back using factors
scaler = MinMaxScaling(
        min_inputs={i: x_data_min[inputs[i]] for i in range(len(inputs))},
        max_inputs={i: x_data_max[inputs[i]] for i in range(len(inputs))},
        min_outputs={i: y_data_min[outputs[i]] for i in range(len(outputs))},
        max_outputs={i: y_data_max[outputs[i]] for i in range(len(outputs))},
    )

# saving unscaled bounds
#input
unscaled_lb = x_data_min[inputs].to_numpy()
unscaled_ub = x_data_max[inputs].to_numpy()
#output
output_lb = y_data_min[outputs].to_numpy()
output_ub = y_data_max[outputs].to_numpy()

# saving scaled bounds
# input
scaled_lb = x_scaled_min[inputs].to_numpy()
scaled_ub = x_scaled_max[inputs].to_numpy()
# output
scaled_output_lb = y_scaled_min[outputs].to_numpy()
scaled_output_ub = y_scaled_max[outputs].to_numpy()

scaled_input_bounds = {i: (scaled_lb[i], scaled_ub[i]) for i in range(len(inputs))}
unscaled_input_bounds = {i: (unscaled_lb[i], unscaled_ub[i]) for i in range(len(inputs))}

In [ ]:
# Optimisation Data
feed_conc_val = 0.5 # ppb
EPA_conc_val = 9.66e-9 # mol/m3
capacity_val = 10 # m3/h
porosity_val = 0.4

# loading saved nn
nn_PFAS_model = keras.models.load_model("GAC_sig_mse(30,30,30)_100k.keras", compile=False)
net = load_keras_sequential(
        nn_PFAS_model, scaling_object=scaler, scaled_input_bounds=scaled_input_bounds, unscaled_input_bounds=unscaled_input_bounds
    )

# Pyomo model with algebraic constraints
def solve_model(objective='cost', eps_emissions=None, feed_conc=feed_conc_val, EPA_conc=EPA_conc_val, capacity=capacity_val, porosity=porosity_val):

    optimization_model = pyo.ConcreteModel()
    
    # Define free variables (Inputs the solver must optimize)
    optimization_model.diameter = pyo.Var(initialize=(unscaled_lb[1]+unscaled_ub[1])/2, bounds=(unscaled_lb[1], unscaled_ub[1]))
    optimization_model.column_height = pyo.Var(initialize=(unscaled_lb[2]+unscaled_ub[2])/2, bounds=(unscaled_lb[2], unscaled_ub[2]))
    optimization_model.porosity = pyo.Var(initialize=(unscaled_lb[3]+unscaled_ub[3])/2, bounds=(unscaled_lb[3], unscaled_ub[3]))
    optimization_model.ads_time = pyo.Var(initialize=(unscaled_lb[4]+unscaled_ub[4])/2, bounds=(unscaled_lb[4], unscaled_ub[4]))
    
    # Output Variables
    optimization_model.total_LCP = pyo.Var(bounds=(output_lb[0], output_ub[0]))
    optimization_model.emissions = pyo.Var(bounds=(output_lb[1], output_ub[1]))
    optimization_model.PFAS_eff_conc = pyo.Var(bounds=(output_lb[2], output_ub[2]))

    # OMLT Block
    optimization_model.nn = OmltBlock()
    formulated_problem = FullSpaceSmoothNNFormulation(net)
    optimization_model.nn.build_formulation(formulated_problem)
    
    # Fix the known inputs directly to shrink the problem size
    optimization_model.nn.inputs[0].fix(capacity)
    optimization_model.nn.inputs[5].fix(feed_conc)

    # Link only the free variables
    optimization_model.con_in_d = pyo.Constraint(expr= optimization_model.diameter == optimization_model.nn.inputs[1])
    optimization_model.con_in_H = pyo.Constraint(expr= optimization_model.column_height == optimization_model.nn.inputs[2])
    optimization_model.con_in_p = pyo.Constraint(expr= optimization_model.porosity == optimization_model.nn.inputs[3])
    optimization_model.con_in_t = pyo.Constraint(expr= optimization_model.ads_time == optimization_model.nn.inputs[4])

    # Link the outputs
    optimization_model.con_out_LCP = pyo.Constraint(expr= optimization_model.total_LCP == optimization_model.nn.outputs[0])
    optimization_model.con_out_emi = pyo.Constraint(expr= optimization_model.emissions == optimization_model.nn.outputs[1])
    optimization_model.con_out_eff = pyo.Constraint(expr= optimization_model.PFAS_eff_conc == optimization_model.nn.outputs[2])

    # Performance Constraints
    optimization_model.epa_limit = pyo.Constraint(expr= optimization_model.PFAS_eff_conc <= pyo.log(EPA_conc + epsilon))
    
    if eps_emissions is not None:
        optimization_model.emissions_con = pyo.Constraint(expr= optimization_model.emissions <= pyo.log(eps_emissions))
    
    # Objective
    if objective == 'cost':
        optimization_model.obj = pyo.Objective(expr= optimization_model.total_LCP, sense=pyo.minimize)
    else:
        optimization_model.obj = pyo.Objective(expr= optimization_model.emissions, sense=pyo.minimize)

    # BARON Configurations
    options = {
    'MaxTime': 3600*3,   # Maximum time in seconds
    'epsr': 0.05,      # relative gap (percentage fraction)
    'epsa': 1e-4,      # Absolute precision
    # 'threads': 4,     # No of worker threads for parallel processing     
    'prlevel': 1,      # 1 = Print normal logs, 0 = silent      
    'dolocal': 1       # 1 = Use local search heuristics to speed up finding bounds     
    }
    
    solver = pyo.SolverFactory('baron') 
    results = solver.solve(optimization_model, tee=True, options=options, keepfiles=True)

    print("# of variables: ", optimization_model.nvariables())
    print("# of constraints: ", optimization_model.nconstraints())

    term_cond = str(results.solver.termination_condition)
    
    # Silently check if the objective has a value
    obj_val = pyo.value(optimization_model.obj, exception=False)
    solution_found = (obj_val is not None)

    if term_cond == 'optimal':
        print("Success: Global optimal solution found")
        status = "Optimal"
    elif term_cond == 'maxTimeLimit' and solution_found:
        print(f"Success: Time limit hit. Feasible solution found (Objective: {obj_val:.4f}).")
        status = "Feasible (Time Limit)"
    else:
        print(f"Solver stopped ('{term_cond}') and NO feasible solution was found.")
        status = "Infeasible"

    def get_val(var):
        # Silently check each variable, return NaN if empty
        v = pyo.value(var, exception=False)
        return v if v is not None else np.nan

    return (capacity,
            get_val(optimization_model.diameter),
            get_val(optimization_model.column_height),
            get_val(optimization_model.porosity),
            get_val(optimization_model.ads_time),
            feed_conc,
            np.exp(get_val(optimization_model.total_LCP)),
            np.exp(get_val(optimization_model.emissions)),
            np.exp(get_val(optimization_model.PFAS_eff_conc) - epsilon),
            status)

In [ ]:
# Solve for Pareto front using epsilon-constraint method
pareto_points = []
solutions = []

# Minimize cost
result_min_cost = solve_model(objective='cost')
Q1, d1, L1, p1, t1, c1, min_cost, max_emi, eff_conc1, status1 = result_min_cost

if status1 == "Infeasible" or (np.isnan(max_emi) and status1 != "Optimal"):
    print("\nWarning: BARON failed to find a mathematical minimum cost point.")
    print("Falling back to the theoretical max emissions from ANN bounds.")
    max_emi = output_ub[1] * 0.99
    min_cost = np.nan

print(f"Min Cost: {min_cost:.4f} $/kg, Corresponding Emissions: {max_emi:.4f} kg CO2/hr ({status1})")

pareto_points.append((min_cost, max_emi))
solutions.append((Q1, d1, L1, p1, t1, c1, min_cost, max_emi, eff_conc1, status1))

# Minimize emissions
result_min_emissions = solve_model(objective='emissions')
Q2, d2, L2, p2, t2, c2, max_cost, min_emi, eff_conc2, status2 = result_min_emissions

if status2 == "Infeasible" or (np.isnan(min_emi) and status2 != "Optimal"):
    print("\nWarning: BARON failed to find a mathematical minimum emissions point.")
    print("Falling back to the theoretical minimum from the ANN training bounds.")
    min_emi = output_lb[1] * 1.01 
    max_cost = np.nan 

print(f"Min Emissions: {min_emi:.4f} kg CO2/hr, Corresponding Cost: {max_cost:.4f} $/kg ({status2})")

# Generate intermediate points using geometric progression
emissions_range = np.geomspace(max_emi, min_emi, 50) 

for id in range(1, len(emissions_range) - 1):
    em = emissions_range[id]
    
    if np.isnan(em):
        continue
        
    print(f"\nSolving for emissions constraint: {em:.4f} kg CO2/hr")
    result = solve_model(objective='cost', eps_emissions=em)
    
    if result is not None:
        capacity, diameter, height, porosity, time, Co, cost, emi, eff, status = result
        if not np.isnan(cost):
            pareto_points.append((cost, emi))
            solutions.append((capacity, diameter, height, porosity, time, Co, cost, emi, eff, status))
        else:
            print(f"No valid solution found for constraint: {em:.4f} kg CO2/hr")

# Append the absolute minimum emissions point at the very end
pareto_points.append((max_cost, min_emi))
solutions.append((Q2, d2, L2, p2, t2, c2, max_cost, min_emi, eff_conc2, status2))

# Save results to Excel
if solutions:
    df = pd.DataFrame(solutions, columns=['capacity', 'diameter', 'column_height', 'porosity', 'ads_time', 'C_o', 'cost', 'emissions', 'PFAS_eff', 'Status'])
    df.to_excel(f'pareto_results_{capacity_val}m3h.xlsx', index=False)
    print(f"\nPareto results saved to 'pareto_results_{capacity_val}m3h.xlsx'")

# Plot Pareto front
if pareto_points:
    # Filter points where both cost and emissions are valid numbers
    plot_points = [(e, c) for c, e in pareto_points if not np.isnan(c) and not np.isnan(e)]
    
    if plot_points:
        emissions_plot, cost_plot = zip(*plot_points)
        plt.figure(figsize=(8, 6))
        plt.scatter(emissions_plot, cost_plot, c='blue', label='Pareto front')
        plt.plot(emissions_plot, cost_plot, 'b--', alpha=0.5)
        plt.xlabel('Emissions (kg CO2/hr)')
        plt.ylabel('Cost ($/kg)')
        plt.title('Cost vs Emissions')
        plt.legend()
        plt.grid(True)
        plt.savefig(f'pareto_results_plot_{capacity_val}m3h.png')
        plt.show()

    print("\nPareto Optimal Solutions:")
    print('capacity', 'diameter', 'column_height', 'porosity', 'ads_time', "C_o", 'cost', 'emissions', 'PFAS_eff', 'Status')
    print("-" * 110)
    for sol in solutions:
        Q, d, L, p, t, C_o, cost, emissions, eff_conc, status = sol
        print(f"{Q:.4f} | {d:.4f} | {L:.4f} | {p:.4f} | {t:.4f} | {C_o:.4f} | {cost:.4f} | {emissions:.4f} | {eff_conc:.4f} | {status}")
else:
    print("No feasible solutions found.")